# **Notebook Walkthrough: Source N - VoteView Member Ideology Scores**

This notebook prepares the **Member Ideology** scores downloaded from VoteView for use in Bridge Grades. We:

- Download the **HS119_members.csv** file from VoteView (Data Type: Member Ideology; Chamber: Both; Congress: 119th; CSV)  
  - URL: https://voteview.com/data  
  - **Date downloaded:** February 16, 2026  
- Clean and normalize the raw `nominate_dim1` scores (absolute distance from ideological center).  
- Split into **House** and **Senate** subsets, merge into our district/member metadata, and compute percentile scores.  
- Export two CSVs:  
  - `bridge_grade_source_n_house_ideology.csv`  
  - `bridge_grade_source_n_senate_ideology.csv`


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
from scipy.stats import norm
import glob
import warnings
from pandas.errors import SettingWithCopyWarning
warnings.simplefilter(action='ignore', category=SettingWithCopyWarning)

## **Load & Inspect Raw Ideology Data**

In [3]:
# Load VoteView ideology scores for 119th Congress
ideology = pd.read_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_N_2026_06_02/HS119_members.csv',
    usecols=['bioguide_id','nominate_dim1']
).copy()

# Preview
ideology.head(10)

,bioguide_id,nominate_dim1
0,NaN,0.403
1,R000575,0.379
2,S001185,-0.402
3,P000609,0.672
4,M001212,0.647
5,S001220,0.602
6,F000481,-0.391
7,A000055,0.405
8,B001323,0.605
9,R000600,0.327


## **Clean & take absolute value from the Raw Scores**

### **Absolute Value & Missing-Value Diagnostics**

In [4]:
# Convert to absolute distance from center
ideology['N_dim1'] = ideology['nominate_dim1'].abs()

# Report missing bioguide_id or score
missing_biog = ideology['bioguide_id'].isna().sum()
missing_score = ideology['N_dim1'].isna().sum()
print(f"Missing bioguide_id: {missing_biog}, Missing scores: {missing_score}")

Missing bioguide_id: 1, Missing scores: 0


### **Fill Missing Scores & Drop Originals**

In [5]:
# Because these two missing people are non-voting members, we will remove the two rows from the dataframe
ideology = ideology[ideology['bioguide_id'].notna()]

# rename the column N_dim 1 to ideology_dist
# Rename for clarity
ideology.rename(columns={'N_dim1':'ideology_dist'}, inplace=True)

## **Split into House & Senate**

We will merge into our master metadata later; for now we separate to compute percentile scores per chamber.

In [6]:
# Read in the 119th Congress data with bioguide ids
files = sorted(glob.glob('/content/drive/Shareddrives/Data Archive/Bridge_Grades/Input_Files/Snapshot_Raw_2026_06_02/Raw_Others_2026_06_02/119th_Congress_20260602.csv'))
latest = files[-1]
df_119 = pd.read_csv(latest)
df_bioguide = df_119.copy()
df_bioguide = df_bioguide[['Name','bioguide_id','Chamber']]
df_bioguide.head()

,Name,bioguide_id,Chamber
0,Laura Friedman,F000483,House
1,Matt Van Epps,V000139,House
2,Cory A. Booker,B001288,Senate
3,Adam B. Schiff,S001150,Senate
4,Alan Armstrong,A000383,Senate


In [7]:
# Merge chamber info onto ideology
ideology = ideology.merge(df_bioguide, on='bioguide_id', how='left')

# Split sets
ideo_house  = ideology.query("Chamber=='House'").copy()
ideo_senate = ideology.query("Chamber=='Senate'").copy()

## **Compute Percentile Scores & Export**

### **House Percentiles & Export**

In [8]:
# Deduplicate if needed
ideo_house = ideo_house.drop_duplicates(subset='bioguide_id')

# Select and save
house_out = ideo_house[['Name','bioguide_id','nominate_dim1','ideology_dist']]
house_out.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_N/Temp_N_House_v1.0.0_2026_06_02.csv',
    index=False
)

In [9]:
house_out.head(10)

,Name,bioguide_id,nominate_dim1,ideology_dist
0,Mike D. Rogers,R000575,0.379,0.379
1,Terri A. Sewell,S001185,-0.402,0.402
2,Gary J. Palmer,P000609,0.672,0.672
3,Barry Moore,M001212,0.647,0.647
4,Dale W. Strong,S001220,0.602,0.602
5,Shomari Figures,F000481,-0.391,0.391
6,Robert B. Aderholt,A000055,0.405,0.405
7,Nicholas J. Begich,B001323,0.605,0.605
8,Aumua Amata Coleman Radewagen,R000600,0.327,0.327
10,Paul A. Gosar,G000565,0.703,0.703


### **Senate Percentiles & Export**

In [10]:
# Deduplicate
ideo_senate = ideo_senate.drop_duplicates(subset='bioguide_id')

# Select and save
senate_out = ideo_senate[['Name','bioguide_id','nominate_dim1','ideology_dist']]
senate_out.to_csv(
    '/content/drive/Shareddrives/Data Archive/Bridge_Grades/Output_Files/Temp_Files/Temp_N/Temp_N_Senate_v1.0.0_2026_06_02.csv',
    index=False
)

In [11]:
senate_out.head(10)

,Name,bioguide_id,nominate_dim1,ideology_dist
449,Tommy Tuberville,T000278,0.936,0.936
450,Katie Boyd Britt,B001319,0.586,0.586
451,Lisa Murkowski,M001153,0.204,0.204
452,Dan Sullivan,S001198,0.469,0.469
453,Ruben Gallego,G000574,-0.357,0.357
454,Mark Kelly,K000377,-0.267,0.267
455,John Boozman,B001236,0.426,0.426
456,Tom Cotton,C001095,0.578,0.578
457,Adam B. Schiff,S001150,-0.353,0.353
458,Alex Padilla,P000145,-0.443,0.443
